In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [ ]:
# Load all trained models
model_dir = Path('../trained_models')

models = {}

# Recommendation model
try:
    models['recommendation'] = {
        'model': joblib.load(model_dir / 'recommendation' / 'rice_variety_model.pkl'),
        'scaler': joblib.load(model_dir / 'recommendation' / 'feature_scaler.pkl'),
        'encoder': joblib.load(model_dir / 'recommendation' / 'label_encoder.pkl')
    }
    print("✓ Recommendation model loaded")
except Exception as e:
    print(f"✗ Recommendation model not found: {e}")

# Soil health model
try:
    models['soil_health'] = {
        'model': joblib.load(model_dir / 'soil' / 'soil_health_model.pkl')
    }
    with open(model_dir / 'soil' / 'parameter_weights.json') as f:
        models['soil_health']['config'] = json.load(f)
    print("✓ Soil health model loaded")
except Exception as e:
    print(f"✗ Soil health model not found: {e}")

# Soil forecast model
try:
    models['soil_forecast'] = {
        'model': joblib.load(model_dir / 'soil' / 'soil_forecast_model.pkl'),
        'scaler': joblib.load(model_dir / 'soil' / 'forecast_scaler.pkl')
    }
    with open(model_dir / 'soil' / 'forecast_config.json') as f:
        models['soil_forecast']['config'] = json.load(f)
    print("✓ Soil forecast model loaded")
except Exception as e:
    print(f"✗ Soil forecast model not found: {e}")

In [ ]:
# Load metrics from all models
all_metrics = {}

# Weather metrics
try:
    with open(model_dir / 'weather' / 'metrics.json') as f:
        all_metrics['weather'] = json.load(f)
except:
    pass

# Soil health metrics
if 'soil_health' in models and 'config' in models['soil_health']:
    all_metrics['soil_health'] = models['soil_health']['config'].get('metrics', {})

# Soil forecast metrics
if 'soil_forecast' in models and 'config' in models['soil_forecast']:
    all_metrics['soil_forecast'] = models['soil_forecast']['config'].get('metrics', {})

print("\nCollected Metrics:")
for model_name, metrics in all_metrics.items():
    print(f"\n{model_name.upper()}:")
    print(json.dumps(metrics, indent=2))

In [ ]:
# Visualize model performance comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Soil forecast R2 scores
if 'soil_forecast' in all_metrics:
    params = list(all_metrics['soil_forecast'].keys())
    r2_scores = [all_metrics['soil_forecast'][p]['r2'] for p in params]
    
    axes[0].bar(params, r2_scores, color='steelblue')
    axes[0].set_title('Soil Forecast R² Scores')
    axes[0].set_ylabel('R² Score')
    axes[0].set_ylim(0, 1)
    axes[0].tick_params(axis='x', rotation=45)

# Soil forecast MAE
if 'soil_forecast' in all_metrics:
    mae_scores = [all_metrics['soil_forecast'][p]['mae'] for p in params]
    
    axes[1].bar(params, mae_scores, color='coral')
    axes[1].set_title('Soil Forecast MAE')
    axes[1].set_ylabel('Mean Absolute Error')
    axes[1].tick_params(axis='x', rotation=45)

# Weather forecast metrics
if 'weather' in all_metrics:
    weather_params = list(all_metrics['weather'].keys())
    weather_mae = [all_metrics['weather'][p].get('mae', 0) for p in weather_params]
    
    axes[2].bar(weather_params, weather_mae, color='forestgreen')
    axes[2].set_title('Weather Forecast MAE')
    axes[2].set_ylabel('Mean Absolute Error')

plt.tight_layout()
plt.show()

In [ ]:
# Test predictions with sample data
sample_soil = {
    'ph': 6.5,
    'nitrogen': 35,
    'phosphorus': 25,
    'potassium': 150,
    'moisture': 55
}

sample_weather = {
    'temperature': 28,
    'humidity': 75,
    'rainfall': 150
}

print("Sample Input Data:")
print(f"Soil: {sample_soil}")
print(f"Weather: {sample_weather}")

In [ ]:
# Test recommendation model
if 'recommendation' in models:
    features = np.array([[
        sample_soil['ph'],
        sample_soil['nitrogen'],
        sample_soil['phosphorus'],
        sample_soil['potassium'],
        sample_soil['moisture'],
        sample_weather['temperature'],
        sample_weather['humidity'],
        sample_weather['rainfall']
    ]])
    
    scaled_features = models['recommendation']['scaler'].transform(features)
    prediction = models['recommendation']['model'].predict(scaled_features)
    probabilities = models['recommendation']['model'].predict_proba(scaled_features)[0]
    
    recommended_variety = models['recommendation']['encoder'].inverse_transform(prediction)[0]
    
    print(f"\nRecommended Variety: {recommended_variety}")
    print("\nTop 3 Recommendations:")
    top_indices = np.argsort(probabilities)[-3:][::-1]
    for idx in top_indices:
        variety = models['recommendation']['encoder'].inverse_transform([idx])[0]
        prob = probabilities[idx]
        print(f"  {variety}: {prob:.2%}")

In [ ]:
# Summary report
print("\n" + "="*60)
print("MODEL EVALUATION SUMMARY")
print("="*60)

print("\n1. RECOMMENDATION MODEL")
print("   - Algorithm: Random Forest Classifier")
print("   - Features: Soil + Weather parameters")
print("   - Output: Rice variety with confidence scores")

print("\n2. WEATHER FORECAST MODEL")
print("   - Algorithm: Facebook Prophet")
print("   - Features: Time series with seasonality")
print("   - Output: Temperature, rainfall, humidity forecasts")

print("\n3. SOIL HEALTH MODEL")
print("   - Algorithm: Weighted scoring + Random Forest")
print("   - Features: pH, NPK, moisture, organic matter")
print("   - Output: Health score (0-100) + recommendations")

print("\n4. SOIL FORECAST MODEL")
print("   - Algorithm: Gradient Boosting (Multi-output)")
print("   - Features: Current soil + weather + time")
print("   - Output: 90-120 day soil condition forecasts")
print("   - Aligned with rice growth stages")